# Validación y Limpieza de Datos
## Proyecto: Salud Colombia - ETL

Este notebook realiza la validación y limpieza de los datasets antes del proceso ETL.

**Datasets:**
- Afiliados al sistema de salud por departamento, municipio y régimen
- IPS públicas y privadas según nivel de atención y capacidad instalada

In [ ]:
import pandas as pd
import numpy as np
import os
import sys

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

print(f"Python: {sys.version}")
print(f"Pandas: {pd.__version__}")
print(f"NumPy: {np.__version__}")

## 1. Carga de Datos

In [ ]:
DATA_DIR = '../data/raw'

AFILIADOS_FILE = 'Número_de_afiliados_por_departamento,_municipio_y_régimen_20260906.csv'
IPS_FILE = 'Relación_de_IPS_públicas_y_privadas_según_el_nivel_de_atención_y_capacidad_instalada_20260906.csv'

print(f"Cargando datasets desde: {DATA_DIR}")
print(f"Afiliados: {AFILIADOS_FILE}")
print(f"IPS: {IPS_FILE}")

In [ ]:
df_afiliados = pd.read_csv(f"{DATA_DIR}/{AFILIADOS_FILE}", dtype=str)
df_ips = pd.read_csv(f"{DATA_DIR}/{IPS_FILE}", dtype=str)

print("Datasets cargados exitosamente")
print(f"Afiliados: {df_afiliados.shape[0]:,} filas, {df_afiliados.shape[1]} columnas")
print(f"IPS: {df_ips.shape[0]:,} filas, {df_ips.shape[1]} columnas")

## 2. Análisis Exploratorio - Dataset Afiliados

In [ ]:
print("=" * 60)
print("DATASET: AFILIADOS AL SISTEMA DE SALUD")
print("=" * 60)

print("\n Primeras 5 filas:")
df_afiliados.head()

In [ ]:
print("\n Información general del dataset:")
df_afiliados.info()

In [ ]:
print("\n Estadísticas descriptivas:")
df_afiliados.describe(include='all')

In [ ]:
print("\n Tipos de datos por columna:")
for col in df_afiliados.columns:
    print(f"  {col}: {df_afiliados[col].dtype} - Ejemplo: '{df_afiliados[col].iloc[0]}'")

### 2.1 Valores Nulos - Afiliados

In [ ]:
print("\n Valores nulos por columna:")
null_afiliados = df_afiliados.isnull().sum()
null_porcentaje = (null_afiliados / len(df_afiliados) * 100).round(2)

df_null_af = pd.DataFrame({
    'Nulos': null_afiliados,
    'Porcentaje': null_porcentaje
})
df_null_af[df_null_af['Nulos'] > 0].sort_values('Porcentaje', ascending=False)

In [ ]:
print("\n Total de valores nulos:", df_afiliados.isnull().sum().sum())
print(" Filas con al menos un nulo:", df_afiliados.isnull().any(axis=1).sum())

### 2.2 Duplicados - Afiliados

In [ ]:
print("\n Duplicados por columna:")
for col in df_afiliados.columns:
    dup_count = df_afiliados[col].duplicated().sum()
    if dup_count > 0:
        print(f"  {col}: {dup_count:,} duplicados")

In [ ]:
duplicados_completos = df_afiliados.duplicated().sum()
print(f"\n Filas completamente duplicadas: {duplicados_completos:,}")
print(f" Porcentaje: {duplicados_completos/len(df_afiliados)*100:.2f}%")

if duplicados_completos > 0:
    print("\n Ejemplo de filas duplicadas:")
    display(df_afiliados[df_afiliados.duplicated(keep=False)].head(10))

### 2.3 Valores Únicos - Afiliados

In [ ]:
print("\n Valores únicos por columna:")
for col in df_afiliados.columns:
    unique_count = df_afiliados[col].nunique()
    print(f"  {col}: {unique_count:,} valores únicos")

In [ ]:
print("\n Distribución de Régimen:")
print(df_afiliados['IDRegimen'].value_counts())

In [ ]:
print("\n Departamentos con más afiliados:")
depto_count = df_afiliados['Departamento'].value_counts().head(10)
print(depto_count)

### 2.4 Análisis de NumPersonas - Afiliados

In [ ]:
print("\n Análisis de NumPersonas:")
print(f"  Tipo actual: {df_afiliados['NumPersonas'].dtype}")
print(f"  Ejemplo de valores: {df_afiliados['NumPersonas'].head(10).tolist()}")

df_afiliados['NumPersonas_Limpio'] = df_afiliados['NumPersonas'].str.replace('.', '', regex=False)
df_afiliados['NumPersonas_Limpio'] = pd.to_numeric(df_afiliados['NumPersonas_Limpio'], errors='coerce')

print(f"\n Después de limpieza:")
print(f"  Valores nulos: {df_afiliados['NumPersonas_Limpio'].isnull().sum()}")
print(f"  Mínimo: {df_afiliados['NumPersonas_Limpio'].min():,.0f}")
print(f"  Máximo: {df_afiliados['NumPersonas_Limpio'].max():,.0f}")
print(f"  Promedio: {df_afiliados['NumPersonas_Limpio'].mean():,.0f}")

## 3. Análisis Exploratorio - Dataset IPS

In [ ]:
print("=" * 60)
print("DATASET: IPS PÚBLICAS Y PRIVADAS")
print("=" * 60)

print("\n Primeras 5 filas:")
df_ips.head()

In [ ]:
print("\n Información general del dataset:")
df_ips.info()

In [ ]:
print("\n Estadísticas descriptivas:")
df_ips.describe(include='all')

In [ ]:
print("\n Tipos de datos por columna:")
for col in df_ips.columns:
    print(f"  {col}: {df_ips[col].dtype} - Ejemplo: '{df_ips[col].iloc[0]}'")

### 3.1 Valores Nulos - IPS

In [ ]:
print("\n Valores nulos por columna:")
null_ips = df_ips.isnull().sum()
null_porcentaje_ips = (null_ips / len(df_ips) * 100).round(2)

df_null_ips = pd.DataFrame({
    'Nulos': null_ips,
    'Porcentaje': null_porcentaje_ips
})
df_null_ips[df_null_ips['Nulos'] > 0].sort_values('Porcentaje', ascending=False)

In [ ]:
print("\n Total de valores nulos:", df_ips.isnull().sum().sum())
print(" Filas con al menos un nulo:", df_ips.isnull().any(axis=1).sum())

### 3.2 Duplicados - IPS

In [ ]:
print("\n Duplicados por columna:")
for col in df_ips.columns:
    dup_count = df_ips[col].duplicated().sum()
    if dup_count > 0:
        print(f"  {col}: {dup_count:,} duplicados")

In [ ]:
duplicados_completos_ips = df_ips.duplicated().sum()
print(f"\n Filas completamente duplicadas: {duplicados_completos_ips:,}")
print(f" Porcentaje: {duplicados_completos_ips/len(df_ips)*100:.2f}%")

if duplicados_completos_ips > 0:
    print("\n Ejemplo de filas duplicadas:")
    display(df_ips[df_ips.duplicated(keep=False)].head(10))

### 3.3 Valores Únicos - IPS

In [ ]:
print("\n Valores únicos por columna:")
for col in df_ips.columns:
    unique_count = df_ips[col].nunique()
    print(f"  {col}: {unique_count:,} valores únicos")

In [ ]:
print("\n Distribución de Naturaleza:")
print(df_ips['naturaleza'].value_counts())

In [ ]:
print("\n Distribución de Nivel de Atención:")
print(df_ips['num nivel atencion'].value_counts())

### 3.4 Análisis de Capacidad Instalada - IPS

In [ ]:
print("\n Análisis de Capacidad Instalada:")
df_ips['capacidad_limpia'] = pd.to_numeric(df_ips['num cantidad capacidad instalada'], errors='coerce')

print(f"  Valores nulos: {df_ips['capacidad_limpia'].isnull().sum()}")
print(f"  Mínimo: {df_ips['capacidad_limpia'].min():,.0f}")
print(f"  Máximo: {df_ips['capacidad_limpia'].max():,.0f}")
print(f"  Promedio: {df_ips['capacidad_limpia'].mean():,.2f}")

In [ ]:
print("\n Distribución por Grupo de Capacidad:")
print(df_ips['nom grupo capacidad '].value_counts())

## 4. Resumen de Calidad de Datos

In [ ]:
print("=" * 60)
print("RESUMEN DE CALIDAD DE DATOS")
print("=" * 60)

print("\n AFILIADOS:")
print(f"  Total registros: {len(df_afiliados):,}")
print(f"  Duplicados completos: {df_afiliados.duplicated().sum():,}")
print(f"  Valores nulos totales: {df_afiliados.isnull().sum().sum():,}")
print(f"  Columnas: {list(df_afiliados.columns)}")

print("\n IPS:")
print(f"  Total registros: {len(df_ips):,}")
print(f"  Duplicados completos: {df_ips.duplicated().sum():,}")
print(f"  Valores nulos totales: {df_ips.isnull().sum().sum():,}")
print(f"  Columnas: {list(df_ips.columns)}")

## 5. Limpieza de Datos

In [ ]:
print("=" * 60)
print("PROCESO DE LIMPIEZA")
print("=" * 60)

df_afiliados_clean = df_afiliados.copy()
df_ips_clean = df_ips.copy()

### 5.1 Limpieza - Afiliados

In [ ]:
print("\n Limpieza del dataset de Afiliados:")
print(f"  Registros iniciales: {len(df_afiliados_clean):,}")

# Eliminar columna temporal
df_afiliados_clean = df_afiliados_clean.drop(columns=['NumPersonas_Limpio'])

# Limpiar NumPersonas
df_afiliados_clean['NumPersonas'] = df_afiliados_clean['NumPersonas'].str.replace('.', '', regex=False)
df_afiliados_clean['NumPersonas'] = pd.to_numeric(df_afiliados_clean['NumPersonas'], errors='coerce').fillna(0).astype(int)

# Convertir tipos numéricos
df_afiliados_clean['Año'] = df_afiliados_clean['Año'].astype(int)
df_afiliados_clean['Mes'] = df_afiliados_clean['Mes'].astype(int)

# Eliminar registros con 0 afiliados
df_afiliados_clean = df_afiliados_clean[df_afiliados_clean['NumPersonas'] > 0]

# Eliminar duplicados
df_afiliados_clean = df_afiliados_clean.drop_duplicates()

print(f"  Registros finales: {len(df_afiliados_clean):,}")
print(f"  Registros eliminados: {len(df_afiliados) - len(df_afiliados_clean):,}")

In [ ]:
print("\n Verificación post-limpieza (Afiliados):")
print(f"  Duplicados: {df_afiliados_clean.duplicated().sum()}")
print(f"  Nulos: {df_afiliados_clean.isnull().sum().sum()}")
print(f"  NumPersonas negativos: {(df_afiliados_clean['NumPersonas'] < 0).sum()}")

### 5.2 Limpieza - IPS

In [ ]:
print("\n Limpieza del dataset de IPS:")
print(f"  Registros iniciales: {len(df_ips_clean):,}")

# Eliminar columna temporal
df_ips_clean = df_ips_clean.drop(columns=['capacidad_limpia'])

# Limpiar NIT (eliminar comas)
df_ips_clean['nit IPS '] = df_ips_clean['nit IPS '].str.replace(',', '', regex=False)

# Convertir num nivel atencion a numérico
df_ips_clean['num nivel atencion'] = pd.to_numeric(df_ips_clean['num nivel atencion'], errors='coerce')

# Limpiar capacidad instalada
df_ips_clean['num cantidad capacidad instalada'] = pd.to_numeric(
    df_ips_clean['num cantidad capacidad instalada'], errors='coerce'
).fillna(0).astype(int)

# Eliminar registros sin código prestador o nombre
df_ips_clean = df_ips_clean.dropna(subset=['Código prestador', 'Nombre prestador'])

# Eliminar duplicados
df_ips_clean = df_ips_clean.drop_duplicates()

print(f"  Registros finales: {len(df_ips_clean):,}")
print(f"  Registros eliminados: {len(df_ips) - len(df_ips_clean):,}")

In [ ]:
print("\n Verificación post-limpieza (IPS):")
print(f"  Duplicados: {df_ips_clean.duplicated().sum()}")
print(f"  Nulos totales: {df_ips_clean.isnull().sum().sum()}")
print(f"  Sin código prestador: {df_ips_clean['Código prestador'].isnull().sum()}")

## 6. Guardar Datos Limpios

In [ ]:
OUTPUT_DIR = '../data/processed'
os.makedirs(OUTPUT_DIR, exist_ok=True)

df_afiliados_clean.to_csv(f"{OUTPUT_DIR}/afiliados_clean.csv", index=False, encoding='utf-8')
df_ips_clean.to_csv(f"{OUTPUT_DIR}/ips_clean.csv", index=False, encoding='utf-8')

print(f"Datos guardados en: {OUTPUT_DIR}")
print(f"  afiliados_clean.csv: {len(df_afiliados_clean):,} registros")
print(f"  ips_clean.csv: {len(df_ips_clean):,} registros")

## 7. Validación Final

In [ ]:
print("=" * 60)
print("VALIDACIÓN FINAL")
print("=" * 60)

print("\n Dataset Afiliados (limpio):")
print(df_afiliados_clean.info())

print("\n" + "=" * 60)
print("\n Dataset IPS (limpio):")
print(df_ips_clean.info())

In [ ]:
print("\n Muestra de datos limpios - Afiliados:")
df_afiliados_clean.head(10)

In [ ]:
print("\n Muestra de datos limpios - IPS:")
df_ips_clean.head(10)

## Resumen

### Resultados de Validación y Limpieza

| Dataset | Registros Originales | Registros Finales | Eliminados |
|---------|---------------------|-------------------|------------|
| Afiliados | {:,} | {:,} | {:,} |
| IPS | {:,} | {:,} | {:,} |

### Acciones Realizadas
1. Conversión de tipos de datos
2. Limpieza de campos numéricos (puntos como separadores de miles)
3. Eliminación de duplicados
4. Eliminación de registros con valores nulos en campos críticos
5. Eliminación de registros con valores inválidos (0 afiliados)
6. Guardado de datos limpios en `data/processed/`